In [ ]:
import os
from getpass import getpass

# Enter your Groq API key securely
GROQ_API_KEY = getpass(" Enter your Groq API Key: ")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("✅ API Key set successfully!")

In [ ]:
import os
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
import warnings
warnings.filterwarnings('ignore')

# --- 1. UI Elements ---
api_input = widgets.Password(description="Groq API Key:", layout=widgets.Layout(width='400px'))
uploader = widgets.FileUpload(accept='.pdf,.txt', multiple=False, description="Upload File")
build_btn = widgets.Button(description="Build RAG Pipeline", button_style='success')
setup_out = widgets.Output()

chat_out = widgets.Output(layout={'border': '1px solid #ccc', 'height': '450px', 'overflow': 'auto', 'padding': '15px'})
chat_input = widgets.Text(placeholder='Ask me anything about your document...', layout=widgets.Layout(width='80%'))
send_btn = widgets.Button(description='Ask', button_style='primary')
rag_chain = None

# --- 2. Build Pipeline Logic ---
def on_build_clicked(b):
    global rag_chain
    with setup_out:
        clear_output()
        if not api_input.value or not uploader.value:
            print(" Please provide both your Groq API Key and upload a document.")
            return
        
        os.environ["GROQ_API_KEY"] = api_input.value
        
        # Save uploaded file
        uploaded_file = uploader.value[0]
        file_name = uploaded_file['name']
        with open(file_name, "wb") as f:
            f.write(uploaded_file['content'])
            
        print(f"📥 Processing '{file_name}'...")
        loader = PyPDFLoader(file_name) if file_name.endswith('.pdf') else TextLoader(file_name, encoding="utf-8")
        pages = loader.load()
        
        # Larger chunk sizes to keep more contextual information intact
        splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
        chunks = splitter.split_documents(pages)
        
        print("🔄 Generating embeddings and vector store...")
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        vector_store = FAISS.from_documents(chunks, embeddings)
        
        print("🤖 Connecting to Groq LLM...")
        llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.3)
        
        # Pulling 8 large chunks at once to scan wide ranges of your 59-page document
        retriever = vector_store.as_retriever(search_kwargs={"k": 8})
        
        # Explicit ChatGPT-style Persona Prompt
        prompt = PromptTemplate(
            template=(
                "System: You are an advanced AI assistant designed to emulate ChatGPT/Gemini. "
                "Your objective is to provide highly detailed, exhaustive, and thoroughly comprehensive answers. "
                "Do not summarize aggressively. Expand on definitions, provide step-by-step explanations, "
                "and structure your response beautifully with descriptive headings, bullet points, and bold text.\n\n"
                "Context from Document:\n{context}\n\n"
                "Question: {question}\n\n"
                "Detailed Markdown Answer:"
            ),
            input_variables=["context", "question"]
        )
        rag_chain = RetrievalQA.from_chain_type(
            llm=llm, chain_type="stuff", retriever=retriever, 
            chain_type_kwargs={"prompt": prompt}
        )
        
        if os.path.exists(file_name):
            os.remove(file_name)
            
        print("✅ Professional-Grade RAG Pipeline Ready!")
        display(widgets.VBox([chat_out, widgets.HBox([chat_input, send_btn])]))

# --- 3. Chat Logic with Markdown Rendering ---
def on_send_clicked(b=None):
    with chat_out:
        q = chat_input.value
        if not q: return
        
        # Renders your input with rich bold formatting
        display(Markdown(f"👤 **You:** {q}"))
        chat_input.value = "" 
        
        try:
            res = rag_chain.invoke({"query": q})
            # Renders the AI response as beautiful ChatGPT-style Markdown
            display(Markdown(f"🤖 **AI:**\n{res['result']}"))
            display(Markdown("---"))
        except Exception as e:
            print(f"Error: {e}")
            display(Markdown("---"))

build_btn.on_click(on_build_clicked)
send_btn.on_click(on_send_clicked)
chat_input.on_submit(on_send_clicked)

print("⚙️ Setup your RAG Chatbot:")
display(widgets.VBox([api_input, uploader, build_btn, setup_out]))